## Setup

In [ ]:
!git clone --branch 3-condition-evaluation https://github.com/GoharKhachatryan/GenAI_project.git

In [ ]:
%cd GenAI_project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ln -s /content/drive/MyDrive/for_bostongene/data /content/GenAI_project/data
!ln -s /content/drive/MyDrive/for_bostongene/dataset.pt /content/GenAI_project/dataset.pt
!ln -s /content/drive/MyDrive/for_bostongene/pca_cache.npz /content/GenAI_project/pca_cache.npz

In [ ]:
%cd GenAI_project

## Condition evaluation

This notebook performs the following operations:

1. Generates samples from the held-out condition sets (a subset of the test set).

2. Extracts features from the genetated samples.

3. Analyzes the distribution of the $||c - \hat{c}||$.

4. Performs a feature-level analysis.

## Import

In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from prepare_dataset import PairedCIFAR10
from pseudo_crossmodal import (
    extract_condition,
    load_pca,
)

from scripts.unet import UNet
from scripts.blocks import (
    ResBlock,
    ScaleShiftResBlock,
)
from scripts.model import (
    ConditionalDenoiser,
    ConditionalDenoiserConcat,
)
from scripts.diffusion import DDPM
from scripts.samplers import (
    ddpm_sample_respaced,
    ddim_sample,
)

## Reproducibility & Configs

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
# Take 100 samples per class,
# since taking the full test set is computationally heavy
NUM_PER_CLASS = 100
NUM_EVAL_SAMPLES = 1000

NUM_STEPS = 100
BATCH_SIZE = 128

DDIM_ETA = 0.0

In [ ]:
CHECKPOINTS_PATH = "/content/drive/MyDrive/for_bostongene/checkpoints/"
CHECKPOINTS = {
    "add_add": CHECKPOINTS_PATH + "add_add/" + "best_model.pt",
    "concat_add": CHECKPOINTS_PATH + "concat_add/" + "best_model.pt",
    "add_scale_shift": CHECKPOINTS_PATH + "add_scale_shift/" + "best_model.pt",
    "concat_scale_shift": CHECKPOINTS_PATH + "concat_scale_shift/" + "best_model.pt",
}

## Create test set

In [ ]:
test_dataset = PairedCIFAR10("test")

print("Test set size:", len(test_dataset))

image, cond, label = test_dataset[0]

print("Image shape:", image.shape)
print("Condition shape:", cond.shape)
print("Label:", label)

In [ ]:
indices_by_class = {
    class_id: []
    for class_id in range(10)
}

for idx in range(len(test_dataset)):
    _, _, label = test_dataset[idx]

    label = int(label)

    if len(indices_by_class[label]) < NUM_PER_CLASS:
        indices_by_class[label].append(idx)

    if all(
        len(v) >= NUM_PER_CLASS
        for v in indices_by_class.values()
    ):
        break

In [ ]:
eval_indices = []

for class_id in range(10):
    eval_indices.extend(
        indices_by_class[class_id]
    )

print("Number of evaluation samples:", len(eval_indices))

In [ ]:
generator = torch.Generator().manual_seed(SEED)

perm = torch.randperm(
    len(eval_indices),
    generator=generator,
).tolist()

eval_indices = [
    eval_indices[i]
    for i in perm
]

## Preprocessing

In [ ]:
eval_images = []
eval_conditions = []
eval_labels = []

for idx in eval_indices:
    image, cond, label = test_dataset[idx]

    eval_images.append(image)
    eval_conditions.append(cond)
    eval_labels.append(label)

eval_images = torch.stack(eval_images)
eval_conditions = torch.stack(eval_conditions)
eval_labels = torch.tensor(eval_labels)

print(eval_images.shape)
print(eval_conditions.shape)
print(eval_labels.shape)

In [ ]:
reference_checkpoint = torch.load(
    CHECKPOINTS["add_add"],
    map_location="cpu",
)

cond_mean = reference_checkpoint["cond_mean"]
cond_std = reference_checkpoint["cond_std"]

print(cond_mean.shape)
print(cond_std.shape)

In [ ]:
eval_conditions_norm = (
    eval_conditions - cond_mean
) / (cond_std + 1e-8)

In [ ]:
pca_state = load_pca()

In [ ]:
generator = torch.Generator().manual_seed(SEED)

# Creats a fixed initial noise for each sample for fair evaluation
x_init_all = torch.randn(
    len(eval_indices),
    3,
    32,
    32,
    generator=generator,
)

## Model

In [ ]:
# Re-use this function instead of manually creating the architecture for each experiment (4)
def build_model(variant):
    if variant == "add_add":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ResBlock,
        )

        model = ConditionalDenoiser(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    elif variant == "concat_add":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ResBlock,
        )

        model = ConditionalDenoiserConcat(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    elif variant == "add_scale_shift":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ScaleShiftResBlock,
        )

        model = ConditionalDenoiser(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    elif variant == "concat_scale_shift":
        unet = UNet(
            in_channels=3,
            out_channels=3,
            base_channels=64,
            emb_dim=256,
            block_cls=ScaleShiftResBlock,
        )

        model = ConditionalDenoiserConcat(
            unet=unet,
            time_dim=256,
            cond_dim=16,
            emb_dim=256,
        )

    else:
        raise ValueError(
            f"Unknown model variant: {variant}"
        )

    return model

In [ ]:
def load_model(variant, device):
    model = build_model(variant)

    checkpoint = torch.load(
        CHECKPOINTS[variant],
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model = model.to(device)
    model.eval()

    return model

In [ ]:
diffusion = DDPM(
    timesteps=1000,
    schedule="cosine",
    device=device,
)

## Cycle metrics

In [ ]:
def compute_cycle_metrics(
    cond_true,
    cond_hat,
    cond_std,
):
    error = cond_true - cond_hat

    raw_l2 = torch.linalg.vector_norm(
        error,
        dim=1,
    )

    standardized_error = (
        error
        / (cond_std + 1e-8)
    )

    standardized_l2 = torch.linalg.vector_norm(
        standardized_error,
        dim=1,
    )

    feature_mae = (
        error.abs().mean(dim=0)
    )

    feature_mae_std = (
        standardized_error.abs().mean(dim=0)
    )

    return {
        "raw_l2": raw_l2,
        "standardized_l2": standardized_l2,
        "feature_mae": feature_mae,
        "feature_mae_std": feature_mae_std,
    }

## Evaluation

In [ ]:
@torch.no_grad()
def evaluate_cycle_consistency(
    model,
    sampler_name,
    cond_original,
    cond_normalized,
    x_init_all,
    cond_std,
    pca_state,
    diffusion,
    batch_size=128,
    num_steps=100,
    eta=0.0,
):
    all_cond_hat = []

    start_time = time.time()

    for start in range(
        0,
        len(cond_original),
        batch_size,
    ):
        end = min(
            start + batch_size,
            len(cond_original),
        )

        cond_batch = (
            cond_normalized[start:end]
            .to(device)
        )

        x_init_batch = (
            x_init_all[start:end]
            .to(device)
        )

        if sampler_name == "ddim":
            generated = ddim_sample(
                model=model,
                ddpm=diffusion,
                cond=cond_batch,
                num_steps=num_steps,
                eta=eta,
                x_init=x_init_batch,
            )

        elif sampler_name == "ddpm":
            # Resetting globally for every batch is NOT what we want.
            # Seed once before the evaluation loop instead.
            generated = ddpm_sample_respaced(
                model=model,
                ddpm=diffusion,
                cond=cond_batch,
                num_steps=num_steps,
                x_init=x_init_batch,
            )

        else:
            raise ValueError(
                f"Unknown sampler: {sampler_name}"
            )

        cond_hat = extract_condition(
            generated,
            pca_state,
        )

        all_cond_hat.append(
            cond_hat.cpu()
        )

    cond_hat = torch.cat(
        all_cond_hat,
        dim=0,
    )

    metrics = compute_cycle_metrics(
        cond_true=cond_original,
        cond_hat=cond_hat,
        cond_std=cond_std,
    )

    metrics["cond_hat"] = cond_hat
    metrics["runtime_sec"] = time.time() - start_time

    return metrics

## Experiments

In [ ]:
# before calling evaluate_cycle_consistency function for a DDPM model, reset the RNG once:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# Since the colab runtime can be interrupted, save the results after each update
RESULTS_PATH = "/content/drive/MyDrive/for_bostongene/cycle_results.pt"

def save_results(results, path=RESULTS_PATH):
    torch.save(results, path)

In [ ]:
import os

if os.path.exists(RESULTS_PATH):
    results = torch.load(
        RESULTS_PATH,
        map_location="cpu",
    )
    print(f"Loaded {len(results)} existing results.")
else:
    results = {}

In [ ]:
model = load_model("add_add", device)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

results[("add_add", "ddim")] = (
    evaluate_cycle_consistency(
        model=model,
        sampler_name="ddim",
        cond_original=eval_conditions,
        cond_normalized=eval_conditions_norm,
        x_init_all=x_init_all,
        cond_std=cond_std,
        pca_state=pca_state,
        diffusion=diffusion,
        batch_size=BATCH_SIZE,
        num_steps=NUM_STEPS,
        eta=DDIM_ETA,
    )
)

save_results(results)

In [ ]:
result = results[("add_add", "ddim")]

print(
    "Mean raw L2:",
    result["raw_l2"].mean().item(),
)

print(
    "Mean standardized L2:",
    result["standardized_l2"].mean().item(),
)

print(
    "Runtime:",
    result["runtime_sec"],
)

## A loop version

In [ ]:
MODEL_VARIANTS = [
    "add_add",
    "concat_add",
    "add_scale_shift",
    "concat_scale_shift",
]

SAMPLERS = [
    "ddpm",
    "ddim",
]

In [ ]:
for variant in MODEL_VARIANTS:
    print(f"\nLoading {variant}")

    model = load_model(variant, device)

    for sampler_name in SAMPLERS:

        key = (variant, sampler_name)

        # For the case in the previous part
        if key in results:
          print(f"Skipping {key}: already computed.")
          continue

        # Skip already-computed configurations
        if key in results:
            print(f"  Skipping {key}: already computed.")
            continue

        print(
            f"  Evaluating {sampler_name}..."
        )

        # Reset RNG once before this complete evaluation
        torch.manual_seed(SEED)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        result = evaluate_cycle_consistency(
            model=model,
            sampler_name=sampler_name,
            cond_original=eval_conditions,
            cond_normalized=eval_conditions_norm,
            x_init_all=x_init_all,
            cond_std=cond_std,
            pca_state=pca_state,
            diffusion=diffusion,
            batch_size=BATCH_SIZE,
            num_steps=NUM_STEPS,
            eta=DDIM_ETA,
        )

        results[key] = result

        # Save immediately after successful completion
        save_results(results)

        print(
            "    standardized L2:",
            result["standardized_l2"]
            .mean()
            .item()
        )

        print(
            "    runtime:",
            round(
                result["runtime_sec"],
                1,
            ),
            "sec",
        )

        print("    Results saved.")

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Build summary from csv

In [ ]:
def build_summary(results):
    rows = []

    for (variant, sampler), result in results.items():
        rows.append({
            "variant": variant,
            "sampler": sampler,
            "mean_raw_l2": result["raw_l2"].mean().item(),
            "std_raw_l2": result["raw_l2"].std().item(),
            "mean_standardized_l2":
                result["standardized_l2"].mean().item(),
            "std_standardized_l2":
                result["standardized_l2"].std().item(),
            "runtime_sec": result["runtime_sec"],
        })

    return pd.DataFrame(rows)

In [ ]:
if os.path.exists(RESULTS_PATH):
    results = torch.load(
        RESULTS_PATH,
        map_location="cpu",
    )
    print(f"Loaded {len(results)} existing results.")

In [ ]:
summary = build_summary(results)

summary.to_csv(
    "/content/drive/MyDrive/for_bostongene/cycle_summary.csv",
    index=False,
)